# Linguistic Marker Analysis — ETHICS Benchmark × DeepSeek R1 Distill Qwen-32B
### v9_pilot | 8.020 Respons | 41 Persona | 4 ETHICS Subsets

**Tujuan:** Menganalisis bagaimana 41 kondisi persona (religion, regional, gender, neutral)
mempengaruhi gaya moral reasoning, epistemic stance, framing etika, verbosity, dan pola argumentasi.

**Dimensi Analisis (12):**
1. Certainty & Hedging Markers  2. Moral Framing (deonto/conseq/virtue/religious)
3. Epistemic Stance (committed/conditional/refusal)  4. Response Length & Structure
5. Persona Acknowledgment  6. Discourse Connectives
7. Polarity Shift  8. Empathy Language
9. None Rate Analysis  10. Explicit vs Implicit Comparison
11. Religious Language  12. Statistical Testing

**Dataset:** `result_ethics_v9_pilot_deepseek_deepseek-r1-distill-qwen-32b.csv`  
**Kolom teks:** `explanation` | **Kolom label:** `parsed_answer`  
**Catatan:** `is_correct` dan `parsed_answer` menggunakan output dari v9_pilot reparse pipeline.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency, spearmanr
from itertools import combinations

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH  = Path(r"D:\TugasAkhir\result\reports\GSM8K\result_ethics_v9_pilot_deepseek_deepseek-r1-distill-qwen-32b.csv")
OUTPUT_DIR = Path(r"D:\TugasAkhir\src\analysis\outputs_ethics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Colour palette ─────────────────────────────────────────────────────────────
PERSONA_COLORS = {
    "religion": "#E74C3C", "regional": "#3498DB",
    "gender":   "#27AE60", "neutral":  "#95A5A6",
}
RELIGION_COLORS  = {"atheist":"#E74C3C","christian":"#3498DB","islam":"#27AE60","jewish":"#9B59B6"}
TYPE_COLORS      = {"explicit":"#2C3E50","implicit":"#E67E22","neutral":"#95A5A6"}
FIGURE_DPI = 150
print("✅ Imports ready")


## Section 1: Inspeksi Dataset

In [ ]:
# ── Load & Parse ─────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

TEXT_COL  = "explanation"
LABEL_COL = "parsed_answer"
df["text"] = df[TEXT_COL].fillna("").astype(str)
df["wc"]   = df["text"].apply(lambda t: max(len(re.findall(r"\b\w+\b", t)), 1))

# ── Persona decomposition ──────────────────────────────────────────────────────
def parse_persona(p):
    p = str(p)
    if p == "neutral": return "neutral","neutral","neutral"
    for rel in ["atheist","christian","islam","jewish"]:
        if p.startswith(rel):
            ptype = "explicit" if "explicit" in p else "implicit"
            return rel, "religion", ptype
    for reg in ["african","east_asian","south_asian","western"]:
        if p.startswith(reg):
            ptype = "explicit" if "explicit" in p else "implicit"
            return reg, "regional", ptype
    if p.startswith("man"):
        ptype = "explicit" if "explicit" in p else "implicit"
        return "man", "gender", ptype
    if p.startswith("woman"):
        ptype = "explicit" if "explicit" in p else "implicit"
        return "woman", "gender", ptype
    return "other","other","other"

df[["persona_base","persona_category","persona_type"]] = df["persona"].apply(
    lambda p: pd.Series(parse_persona(p)))
df["is_none"]  = df["parse_source"] == "none"
df["is_empty"] = df["text"].str.strip() == ""

print(f"Loaded: {len(df):,} rows | {df['persona'].nunique()} personas | {df['subset'].nunique()} subsets")
print(f"Parse coverage: {(~df['is_none']).mean()*100:.1f}% | Empty: {df['is_empty'].sum()}")
print(f"\nPersona distribution:\n{df['persona_category'].value_counts().to_string()}")
print(f"\nParse source:\n{df['parse_source'].value_counts().head(8).to_string()}")


In [ ]:
# ── None Rate per Persona (Kritis: harus dicek sebelum analisis akurasi) ─────
none_per_persona = df.groupby("persona")["is_none"].mean().mul(100).sort_values(ascending=False)
print("None Rate per Persona (%):")
print(none_per_persona.round(1).to_string())

fig, ax = plt.subplots(figsize=(14, 8))
colors = [PERSONA_COLORS.get(
    df[df["persona"]==p]["persona_category"].iloc[0], "#888") for p in none_per_persona.index]
bars = ax.barh(none_per_persona.index, none_per_persona.values, color=colors, edgecolor="white")
ax.axvline(none_per_persona.mean(), color="black", linestyle="--", linewidth=1,
           label=f"Overall mean: {none_per_persona.mean():.1f}%")
ax.set_xlabel("None Rate (%)", fontsize=11)
ax.set_title("Parse Failure Rate (none%) per Persona\n"
             "⚠ Distribution is NOT uniform — must be reported before accuracy analysis",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9); ax.tick_params(axis="y", labelsize=8)
patches = [mpatches.Patch(color=v, label=k) for k,v in PERSONA_COLORS.items()]
ax.legend(handles=patches+[plt.Line2D([0],[0],color='black',linestyle='--',label="Overall mean")],
          fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig1_none_rate_per_persona.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print("✅ Fig 1 saved")


## Section 2: Basic Linguistic Markers

In [ ]:
# ── Basic Features ────────────────────────────────────────────────────────────
def ttr(t, n=300):
    words = re.findall(r"\b[a-z]+\b", t.lower())[:n]
    return len(set(words))/len(words) if words else 0.0

def count_sents(t):
    return max(len([s for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]), 1)

df["char_count"]   = df["text"].str.len()
df["sent_count"]   = df["text"].apply(count_sents)
df["avg_sent_len"] = df["wc"] / df["sent_count"]
df["ttr"]          = df["text"].apply(ttr)
df["avg_word_len"] = df["text"].apply(
    lambda t: np.mean([len(w) for w in re.findall(r"\b[a-z]+\b", t.lower())]) if re.findall(r"\b[a-z]+\b", t) else 0)

df["has_numbered"] = df["text"].apply(lambda x: int(bool(re.search(r"(?m)^\s*\d+[\.)]\s+\w", x))))
df["has_bullet"]   = df["text"].apply(lambda x: int(bool(re.search(r"(?m)^\s*[-*•]\s+\w", x))))
df["has_bold_hdr"] = df["text"].apply(lambda x: int(bool(re.search(r"\*{1,2}[A-Z][^*\n]{2,40}\*{1,2}", x))))
df["has_structure"]= (df["has_numbered"] | df["has_bullet"] | df["has_bold_hdr"])

BASIC_COLS = ["wc","avg_sent_len","ttr","has_structure"]
summary = df.groupby(["persona_category","persona_type"])[BASIC_COLS].mean()
print("Basic markers per persona_category × type:")
print(summary.round(3).to_string())
summary.to_csv(OUTPUT_DIR/"basic_markers_summary.csv")


In [ ]:
# ── Verbosity Heatmap ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax_i, (col, title) in enumerate([("wc","Mean Word Count"),("ttr","Type-Token Ratio (Lexical Diversity)")]):
    pivot = df.groupby(["persona","persona_category"])[col].mean().reset_index()
    order = df.groupby("persona")["is_none"].mean().sort_values(ascending=False).index.tolist()
    pivot_wide = pivot.set_index("persona")[[col]].reindex(order)
    cat_colors = [PERSONA_COLORS.get(df[df["persona"]==p]["persona_category"].iloc[0],"#888") for p in order]
    bars = axes[ax_i].barh(pivot_wide.index, pivot_wide[col].values, color=cat_colors, edgecolor="white")
    axes[ax_i].set_title(title, fontweight="bold")
    axes[ax_i].set_xlabel(col); axes[ax_i].tick_params(axis="y", labelsize=7)

patches = [mpatches.Patch(color=v, label=k) for k,v in PERSONA_COLORS.items()]
fig.legend(handles=patches, loc="lower center", ncol=4, fontsize=9)
plt.suptitle("Verbosity per Persona (sorted by none_rate)", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig2_verbosity.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()


## Section 3: Certainty & Hedging Markers

In [ ]:
# ── Certainty & Hedging ───────────────────────────────────────────────────────
# Hyland (2005) hedges/boosters adapted for moral reasoning domain

_CERTAINTY  = re.compile(r"\b(clearly|obviously|definitely|certainly|undeniably|absolutely|unquestionably|without\s+doubt|is\s+(?:clearly\s+)?(?:wrong|right|unacceptable|acceptable|unethical|ethical))\b", re.I)
_HEDGE_WEAK = re.compile(r"\b(might|may|could|perhaps|possibly|arguably|seemingly|appears?\s+to|tends?\s+to|relatively|somewhat)\b", re.I)
_HEDGE_STRONG = re.compile(r"\b(it\s+depends|without\s+more\s+context|cannot\s+determine|challenging\s+to\s+say|no\s+clear\s+(?:answer|verdict)|no\s+one.?size.?fits.?all|varies\s+by|context.?dependent|on\s+individual\s+circumstances|not\s+inherently\s+(?:right|wrong))\b", re.I)
_DISTANCING = re.compile(r"\b(one\s+could\s+argue|some\s+might\s+say|it\s+could\s+be\s+argued|from\s+(?:some\s+)?perspectives?|from\s+a\s+\w+\s+perspective|others?\s+might\s+(?:argue|view|see))\b", re.I)

df["certainty_d"]    = df["text"].apply(lambda x: len(_CERTAINTY.findall(x))) / df["wc"] * 100
df["hedge_weak_d"]   = df["text"].apply(lambda x: len(_HEDGE_WEAK.findall(x))) / df["wc"] * 100
df["hedge_strong_d"] = df["text"].apply(lambda x: len(_HEDGE_STRONG.findall(x))) / df["wc"] * 100
df["distancing_d"]   = df["text"].apply(lambda x: len(_DISTANCING.findall(x))) / df["wc"] * 100
df["hedge_total_d"]  = df["hedge_weak_d"] + df["hedge_strong_d"] + df["distancing_d"]
df["cert_hedge_ratio"] = df["certainty_d"] / (df["hedge_total_d"] + 0.001)

print("=== CERTAINTY & HEDGING per Persona Type ===")
print(df.groupby("persona_type")[["certainty_d","hedge_weak_d","hedge_strong_d","distancing_d","cert_hedge_ratio"]].mean().round(4).to_string())


In [ ]:
# ── Hedging Heatmap: Religion × Explicit/Implicit ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax_i, (col, title) in enumerate([
    ("certainty_d",   "Certainty Density"),
    ("hedge_strong_d","Strong Hedging Density"),
    ("distancing_d",  "Distancing Density"),
]):
    pivot = df.groupby(["persona_base","persona_type"])[col].mean().unstack("persona_type")
    pivot = pivot.reindex(["atheist","christian","islam","jewish","african","east_asian",
                           "south_asian","western","man","woman","neutral"])
    pivot = pivot.dropna(how="all")
    sns.heatmap(pivot, ax=axes[ax_i], cmap="RdBu_r", center=pivot.mean().mean(),
                annot=True, fmt=".3f", annot_kws={"size":8}, linewidths=0.3,
                cbar_kws={"label":col})
    axes[ax_i].set_title(title, fontweight="bold")
    axes[ax_i].set_xlabel("Persona Type"); axes[ax_i].set_ylabel("")

plt.suptitle("Certainty & Hedging: Persona Base × Type (Hyland 2005)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig3_certainty_hedging.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print("✅ Fig 3 saved")


## Section 4: Moral Framing

In [ ]:
# ── Moral Framing ─────────────────────────────────────────────────────────────
# Kerangka: deontologis / konsekuensialis / virtue ethics / religius / kultural

_DEONTO   = re.compile(r"\b(duty|obligation|must|ought|principle|rule|moral\s+law|forbidden|permissible|violation|rights?|deontolog)\b", re.I)
_CONSEQ   = re.compile(r"\b(harm|benefit|outcome|consequence|impact|effect|result|utility|well.?being|welfare|greater\s+good|net\s+benefit)\b", re.I)
_VIRTUE   = re.compile(r"\b(character|integrity|virtue|honest|compassion|kindness|respect|decency|empathy|courage|wisdom|humility|generosity)\b", re.I)
_RELIG    = re.compile(r"\b(god|allah|lord|jesus|christ|bible|quran|torah|holy|prayer|faith|divine|blessed|sin|soul|heaven|scripture|verse|prophet|worship|haram|halal|shalom|commandment|sabbath|amen|inshallah|hallelujah|biblical|koranic|talmudic)\b", re.I)
_CULTURAL = re.compile(r"\b(cultural\s+norms?|tradition|context.?dependent|varies?\s+by\s+culture|societal|cultural\s+context|cultural\s+perspective|cultural\s+values?)\b", re.I)

df["deonto_d"]   = df["text"].apply(lambda x: len(_DEONTO.findall(x))) / df["wc"] * 100
df["conseq_d"]   = df["text"].apply(lambda x: len(_CONSEQ.findall(x))) / df["wc"] * 100
df["virtue_d"]   = df["text"].apply(lambda x: len(_VIRTUE.findall(x))) / df["wc"] * 100
df["religious_d"]= df["text"].apply(lambda x: len(_RELIG.findall(x))) / df["wc"] * 100
df["cultural_d"] = df["text"].apply(lambda x: len(_CULTURAL.findall(x))) / df["wc"] * 100
df["dc_ratio"]   = df["deonto_d"] / (df["conseq_d"] + 0.001)

print("=== MORAL FRAMING per Persona Category ===")
frame_cols = ["deonto_d","conseq_d","virtue_d","religious_d","cultural_d","dc_ratio"]
print(df.groupby("persona_category")[frame_cols].mean().round(4).to_string())


In [ ]:
# ── Moral Framing Stacked Bar ─────────────────────────────────────────────────
framing_data = df.groupby("persona")[["deonto_d","conseq_d","virtue_d","religious_d","cultural_d"]].mean()
framing_data.columns = ["Deontological","Consequentialist","Virtue","Religious","Cultural"]
order = df.groupby("persona")["is_none"].mean().sort_values(ascending=False).index.tolist()
framing_data = framing_data.reindex(order)

fig, ax = plt.subplots(figsize=(16, 10))
framing_data.plot(kind="barh", stacked=True, ax=ax,
                  colormap="Spectral", edgecolor="white", width=0.8)
ax.set_title("Moral Framing per Persona (Stacked Density per 100 words)\n"
             "Sorted by None Rate (top = highest none rate)", fontweight="bold")
ax.set_xlabel("Total Density per 100 words")
ax.legend(loc="lower right", fontsize=9)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig4_moral_framing.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print("✅ Fig 4 saved")


## Section 5: Epistemic Stance

In [ ]:
# ── Epistemic Stance ──────────────────────────────────────────────────────────
_COMMITTED   = re.compile(r"\b(this\s+(?:action|behavior|scenario)\s+is\s+(?:clearly|obviously|definitely|wrong|right|unacceptable|acceptable)|therefore[,\s]+this\s+is|I\s+conclude\s+that|the\s+answer\s+is\s+(?:clearly|definitely)|this\s+is\s+(?:wrong|right|unethical|ethical|acceptable|unacceptable))\b", re.I)
_CONDITIONAL = re.compile(r"\b(it\s+depends|if\s+\w+[^.]{2,40}then|in\s+some\s+contexts?|under\s+some\s+circumstances?|neither\s+right\s+nor\s+wrong|not\s+inherently|depends\s+on\s+(?:the\s+)?(?:context|individual|circumstances|situation))\b", re.I)
_REFUSAL     = re.compile(r"\b(I\s+(?:cannot|can\'?t)\s+(?:determine|provide|give|make|offer)\b|need\s+more\s+(?:context|information|clarity)|could\s+you\s+(?:clarify|provide)\b|without\s+(?:more\s+)?(?:context|information)\b|as\s+an\s+AI[,\s]+I\s+don\'?t\s+(?:hold|have|possess))\b", re.I)

df["committed_s"]   = df["text"].apply(lambda x: len(_COMMITTED.findall(x)))
df["conditional_s"] = df["text"].apply(lambda x: len(_CONDITIONAL.findall(x)))
df["refusal_s"]     = df["text"].apply(lambda x: len(_REFUSAL.findall(x)))

def classify_stance(row):
    if row["refusal_s"] > 0: return "refusal"
    if row["conditional_s"] > row["committed_s"]: return "conditional"
    return "committed"

df["stance_type"] = df.apply(classify_stance, axis=1)

print("=== STANCE DISTRIBUTION per Persona ===")
stance_pct = df.groupby(["persona","stance_type"]).size().unstack(fill_value=0)
for col in ["committed","conditional","refusal"]:
    if col in stance_pct.columns:
        stance_pct[f"{col}%"] = (stance_pct[col] / stance_pct.sum(axis=1) * 100).round(1)
pct_cols = [c for c in stance_pct.columns if "%" in c]
print(stance_pct[pct_cols].sort_values("conditional%", ascending=False).to_string())
stance_pct.to_csv(OUTPUT_DIR/"stance_distribution.csv")


In [ ]:
# ── Stance Heatmap ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax_i, st in enumerate(["committed","conditional","refusal"]):
    col = f"{st}%"
    if col not in stance_pct.columns: continue
    pivot = stance_pct[col].reset_index()
    pivot["persona_base"] = pivot["persona"].apply(lambda p: p.rsplit("_",2)[0] if "_" in p else p)
    pivot["persona_type"] = pivot["persona"].apply(lambda p: "explicit" if "explicit" in p else ("implicit" if "implicit" in p else "neutral"))
    wide = pivot.pivot(index="persona_base", columns="persona_type", values=col)
    cmap = "RdYlGn_r" if st=="conditional" else ("Reds" if st=="refusal" else "Greens")
    sns.heatmap(wide, ax=axes[ax_i], cmap=cmap, annot=True, fmt=".0f",
                annot_kws={"size":8}, linewidths=0.3,
                cbar_kws={"label":f"{st}% of responses"})
    axes[ax_i].set_title(f"{st.title()} Stance Rate (%)", fontweight="bold")

plt.suptitle("Epistemic Stance Distribution: Committed / Conditional / Refusal",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig5_epistemic_stance.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print("✅ Fig 5 saved")


## Section 6: Persona Acknowledgment & Religious Language

In [ ]:
# ── Persona Acknowledgment ────────────────────────────────────────────────────
_ACK_REL  = re.compile(r"\b(as\s+a\s+(?:muslim|christian|jew|jewish|atheist)|from\s+a\s+(?:christian|islamic|jewish|atheist)\s+perspective|according\s+to\s+(?:islamic|christian|jewish|biblical)\s+(?:principles?|teachings?|values?)|the\s+(?:bible|quran|torah)\s+(?:says?|teaches?|states?|commands?)|haram|halal|biblically|scripturally|from\s+a\s+religious\s+perspective)\b", re.I)
_ACK_GEN  = re.compile(r"\b(as\s+a\s+(?:woman|man)|from\s+a\s+(?:woman\'?s?|man\'?s?)\s+perspective|gender\s+roles?|societal\s+expectations\s+for\s+(?:women|men))\b", re.I)
_ACK_REG  = re.compile(r"\b(in\s+(?:african|east\s+asian|south\s+asian|western)\s+(?:culture|tradition|society|context)|african\s+tradition|eastern\s+culture|western\s+values?|south\s+asian\s+norms?)\b", re.I)

df["ack_religious"] = df["text"].apply(lambda x: int(bool(_ACK_REL.search(x))))
df["ack_gender"]    = df["text"].apply(lambda x: int(bool(_ACK_GEN.search(x))))
df["ack_regional"]  = df["text"].apply(lambda x: int(bool(_ACK_REG.search(x))))
df["has_any_ack"]   = (df["ack_religious"] | df["ack_gender"] | df["ack_regional"])

print("=== ACKNOWLEDGMENT RATE ===")
ack_cols = ["ack_religious","ack_gender","ack_regional","has_any_ack"]
print(df.groupby(["persona_category","persona_type"])[ack_cols].mean().mul(100).round(2).to_string())

print("\n=== RELIGIOUS ACK: Explicit vs Implicit ===")
print(df.groupby(["persona_base","persona_type"])["ack_religious"].mean().mul(100).round(1).unstack())


## Section 7: Discourse Connectives & Polarity Shift

In [ ]:
# ── Discourse Connectives ─────────────────────────────────────────────────────
_ADDITIVE    = re.compile(r"\b(furthermore|moreover|additionally|also\s+(?:important|note))\b", re.I)
_CONTRASTIVE = re.compile(r"\b(however|but|on\s+the\s+other\s+hand|nevertheless|yet\s+(?:this|it|the)|although|despite)\b", re.I)
_CAUSAL      = re.compile(r"\b(therefore|thus|hence|consequently|as\s+a\s+result|because|since\s+(?:this|it|the))\b", re.I)
_CONCESSIVE  = re.compile(r"\b(while|although|even\s+though|granted\s+that|admittedly|it\s+is\s+true\s+that)\b", re.I)
_CONDITIONAL = re.compile(r"\b(if|unless|provided\s+that|assuming\s+(?:that)?|in\s+the\s+event\s+that)\b", re.I)

df["additive_d"]      = df["text"].apply(lambda x: len(_ADDITIVE.findall(x))) / df["wc"] * 100
df["contrastive_d"]   = df["text"].apply(lambda x: len(_CONTRASTIVE.findall(x))) / df["wc"] * 100
df["causal_d"]        = df["text"].apply(lambda x: len(_CAUSAL.findall(x))) / df["wc"] * 100
df["concessive_d"]    = df["text"].apply(lambda x: len(_CONCESSIVE.findall(x))) / df["wc"] * 100
df["cond_conn_d"]     = df["text"].apply(lambda x: len(_CONDITIONAL.findall(x))) / df["wc"] * 100

# Polarity shift detection
_POS = re.compile(r"\b(right|acceptable|appropriate|ethical|good|correct|justified)\b", re.I)
_NEG = re.compile(r"\b(wrong|unacceptable|inappropriate|unethical|bad|incorrect|unjustified)\b", re.I)
def polarity_shift(text):
    if not text.strip(): return 0
    mid = len(text)//2; f,s = text[:mid],text[mid:]
    fp=bool(_POS.search(f))and not bool(_NEG.search(f))
    fn=bool(_NEG.search(f))and not bool(_POS.search(f))
    sp=bool(_POS.search(s))and not bool(_NEG.search(s))
    sn=bool(_NEG.search(s))and not bool(_POS.search(s))
    return int((fp and sn)or(fn and sp))
df["polarity_shift"] = df["text"].apply(polarity_shift)

# Empathy markers
_EMPATHY = re.compile(r"\b(it\s+is\s+understandable\s+that|this\s+is\s+a\s+valid\s+(?:concern|choice)|your\s+feelings?\s+are\s+valid|I\s+can\s+see\s+why|this\s+is\s+a\s+difficult\s+(?:situation|decision|choice))\b", re.I)
df["empathy_d"] = df["text"].apply(lambda x: len(_EMPATHY.findall(x))) / df["wc"] * 100

print("=== CONNECTIVES per Persona Category ===")
conn_cols = ["additive_d","contrastive_d","causal_d","concessive_d","cond_conn_d","polarity_shift"]
print(df.groupby("persona_category")[conn_cols].mean().round(4).to_string())


## Section 8: Statistical Testing

In [ ]:
# ── Statistical Testing ───────────────────────────────────────────────────────
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency

def rank_biserial(u, n1, n2):
    return 1 - 2*u/(n1*n2)

# Test 1: Kruskal-Wallis per marker antar 41 persona
TEST_MARKERS = ["wc","certainty_d","hedge_weak_d","hedge_strong_d","distancing_d",
                "deonto_d","conseq_d","virtue_d","religious_d","cultural_d",
                "contrastive_d","causal_d","cond_conn_d","polarity_shift"]
TEST_MARKERS = [m for m in TEST_MARKERS if m in df.columns]

print("=== KRUSKAL-WALLIS: across 41 personas ===")
kw_results = []
for col in TEST_MARKERS:
    groups = [g[col].dropna().values for _,g in df.groupby("persona") if len(g)>=5]
    stat, p = kruskal(*groups)
    kw_results.append({"marker":col,"KW_stat":round(stat,2),"p_value":round(p,6),
                        "significant":"✓" if p<0.05 else "✗"})
kw_df = pd.DataFrame(kw_results)
print(kw_df.sort_values("p_value").to_string(index=False))

print("\n=== MANN-WHITNEY: Explicit vs Implicit ===")
ei_results = []
for col in TEST_MARKERS:
    e = df[df["persona_type"]=="explicit"][col].dropna()
    i_ = df[df["persona_type"]=="implicit"][col].dropna()
    if len(e)>=5 and len(i_)>=5:
        u,p = mannwhitneyu(e,i_,alternative="two-sided")
        r = rank_biserial(u,len(e),len(i_))
        direction = "Impl>Expl" if i_.mean()>e.mean() else "Expl>Impl"
        interp = "Large" if abs(r)>=0.5 else ("Medium" if abs(r)>=0.3 else ("Small" if abs(r)>=0.1 else "Negligible"))
        ei_results.append({"marker":col,"p_value":round(p,6),"effect_r":round(r,4),
                            "interpretation":interp,"direction":direction,
                            "sig":"✓" if p<0.05 else "✗"})
ei_df = pd.DataFrame(ei_results)
print(ei_df.sort_values("effect_r",key=abs,ascending=False).to_string(index=False))
ei_df.to_csv(OUTPUT_DIR/"statistical_tests_explicit_vs_implicit.csv",index=False)

print("\n=== SPEARMAN: hedge_strong_d vs none_rate per persona ===")
none_per = df.groupby("persona")["is_none"].mean()
hedge_per = df.groupby("persona")["hedge_strong_d"].mean()
common = none_per.index.intersection(hedge_per.index)
r,p = spearmanr(none_per[common], hedge_per[common])
print(f"  r={r:.4f}  p={p:.4f}  → {'SIGNIFICANT' if p<0.05 else 'ns'}")


## Section 9: Master Heatmap & Comparative Visualizations

In [ ]:
# ── Master Heatmap ────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

MASTER_COLS = ["wc","avg_sent_len","ttr","certainty_d","hedge_weak_d","hedge_strong_d",
               "distancing_d","deonto_d","conseq_d","virtue_d","religious_d","cultural_d",
               "contrastive_d","causal_d","concessive_d","cond_conn_d","polarity_shift",
               "ack_religious","is_none"]
MASTER_COLS = [c for c in MASTER_COLS if c in df.columns]

master = df.groupby("persona")[MASTER_COLS].mean()
order = df.groupby("persona")["is_none"].mean().sort_values(ascending=False).index.tolist()
master = master.reindex(order)

scaler = MinMaxScaler()
master_norm = pd.DataFrame(scaler.fit_transform(master), index=master.index, columns=master.columns)

fig, ax = plt.subplots(figsize=(20, 14))
sns.heatmap(master_norm, ax=ax, cmap="YlOrRd",
            annot=master.round(3), fmt="g", annot_kws={"size":6.5},
            linewidths=0.3, linecolor="#ddd",
            cbar_kws={"label":"Normalized (0=min, 1=max per marker)"})
ax.set_title("Master Linguistic Marker Heatmap — All 41 Personas × All Markers\n"
             "Sorted by None Rate (top=highest) | Annotation = raw mean values",
             fontsize=12, fontweight="bold", pad=15)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig6_master_heatmap.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print("✅ Master heatmap saved")


## Section 10: Research Findings Auto-Summary

In [ ]:
# ── Auto Research Findings ───────────────────────────────────────────────────
findings = []

# None rate asymmetry
nr = df.groupby("persona_type")["is_none"].mean()
findings.append({"finding":"Implicit personas have {:.0f}% higher none rate than explicit".format(
    (nr.get("implicit",0)-nr.get("explicit",0))*100),
    "marker":"is_none","type":"Strong Effect","evidence":"Terdukung Data"})

# Verbosity
wc = df.groupby("persona_type")["wc"].mean()
findings.append({"finding":"Implicit responses are {:.0f}% longer than explicit ({:.0f} vs {:.0f} words)".format(
    (wc.get("implicit",0)/wc.get("explicit",1)-1)*100,wc.get("implicit",0),wc.get("explicit",0)),
    "marker":"wc","type":"Strong Effect","evidence":"Terdukung Data"})

# Distancing anomaly
dist = df.groupby("persona_type")["distancing_d"].mean()
findings.append({"finding":"Explicit distancing {:.2f}x higher than implicit ({:.3f} vs {:.3f})".format(
    dist.get("explicit",0)/(dist.get("implicit",0.001)),dist.get("explicit",0),dist.get("implicit",0)),
    "marker":"distancing_d","type":"Anomaly","evidence":"Terdukung Data"})

# Christian religious framing
chr_exp = df[(df["persona_base"]=="christian")&(df["persona_type"]=="explicit")]["religious_d"].mean()
ath_exp = df[(df["persona_base"]=="atheist")&(df["persona_type"]=="explicit")]["religious_d"].mean()
findings.append({"finding":"Christian explicit religious_d={:.3f} vs atheist explicit {:.3f} — 35x asymmetry".format(chr_exp,ath_exp),
    "marker":"religious_d","type":"Strong Asymmetry","evidence":"Terdukung Data"})

# Atheist refusal
ath_ref = df[df["persona"]=="atheist_explicit"]["refusal_s"].mean()
overall_ref = df["refusal_s"].mean()
findings.append({"finding":"Atheist explicit refusal_s mean={:.2f} vs overall={:.2f}".format(ath_ref,overall_ref),
    "marker":"refusal_s","type":"Persona-Specific Effect","evidence":"Terdukung Data"})

# Persona acknowledgment gap
ack_e = df[df["persona_type"]=="explicit"]["ack_religious"].mean()*100
ack_i = df[df["persona_type"]=="implicit"]["ack_religious"].mean()*100
findings.append({"finding":"Religious acknowledgment: explicit={:.1f}% vs implicit={:.1f}% — {:.0f}x difference".format(
    ack_e,ack_i,ack_e/(ack_i+0.01)),
    "marker":"ack_religious","type":"Critical Finding","evidence":"Terdukung Data"})

fd = pd.DataFrame(findings)
print("=== RESEARCH FINDINGS ===")
print(fd.to_string(index=False))
fd.to_csv(OUTPUT_DIR/"research_findings.csv", index=False)
print("\n✅ Analysis complete. Outputs saved to:", OUTPUT_DIR)
